# Lab 4: 종합 벤치마크 — 세 가지 RAG 비교

**학습 목표**
- Standard RAG, LightRAG, Karpathy Wiki를 동일 질문으로 비교
- LLM-as-Judge 자동 평가 체험
- 질문 유형별 (팩트, 종합, 집계) 강약점 분석

**소요 시간**: 20-30분 (캐시 활용 시)  
**선행 과제**: Lab 1, 2, 3

In [ ]:
import sys, time
sys.path.insert(0, '..')

import numpy as np
from rag_lab import (
    EmbeddingEngine, StandardRAG, LightRAG, KarpathyWiki,
    Evaluator, load_papers,
)

## 1. 시스템 구축

In [ ]:
papers = load_papers()
engine = EmbeddingEngine()

build_times = {}

t0 = time.time()
rag = StandardRAG(papers, engine)
build_times['Standard RAG'] = time.time() - t0

t0 = time.time()
lrag = LightRAG(papers, engine)
build_times['LightRAG'] = time.time() - t0

t0 = time.time()
wiki = KarpathyWiki(papers)
build_times['Karpathy Wiki'] = time.time() - t0

print("구축 시간:")
for name, t in build_times.items():
    print(f"  {name}: {t:.1f}초")

## 2. 벤치마크 질문 정의

3가지 유형의 질문으로 테스트합니다:
- **Factual**: 단일 논문에서 사실 찾기 (쉬움)
- **Synthesis**: 여러 논문의 내용을 연결 (어려움)
- **Aggregation**: 전체 논문에서 정보 수집 (중간)

In [ ]:
questions = [
    {
        "id": "Q1", "type": "factual", "difficulty": "easy",
        "question": "What is the main finding about intergenerational mobility by Card et al.?",
    },
    {
        "id": "Q2", "type": "factual", "difficulty": "easy",
        "question": "How do negative interest rates affect bank performance?",
    },
    {
        "id": "Q3", "type": "synthesis", "difficulty": "hard",
        "question": "Compare the methodological approaches across human capital and auction papers.",
    },
    {
        "id": "Q4", "type": "synthesis", "difficulty": "hard",
        "question": "How does employer credit checking relate to information asymmetry themes?",
    },
    {
        "id": "Q5", "type": "aggregation", "difficulty": "medium",
        "question": "What datasets are used across all papers? Natural experiments vs structural models?",
    },
]

## 3. 벤치마크 실행

In [ ]:
methods = {
    "Standard RAG": rag,
    "LightRAG": lrag,
    "Karpathy Wiki": wiki,
}
evaluator = Evaluator()
all_results = []

for q in questions:
    print(f"\n{'━' * 55}")
    print(f"[{q['id']}] {q['question'][:65]}...")
    print(f"  유형: {q['type']} | 난이도: {q['difficulty']}")
    
    q_result = {"question": q, "scores": {}}
    answers = {}
    
    for name, method in methods.items():
        result = method.query(q["question"])
        answers[name] = result.answer
        q_result[name] = {
            "answer": result.answer,
            "time": result.total_time,
            "context_length": result.context_length,
        }
    
    scores = evaluator.compare(q["question"], answers, q["type"])
    for name, s in scores.items():
        q_result["scores"][name] = s
    
    evaluator.print_comparison(scores)
    all_results.append(q_result)

## 4. 결과 분석

In [ ]:
# 전체 평균 점수
print("\n" + "=" * 60)
print("  전체 결과 요약")
print("=" * 60)

method_names = list(methods.keys())
dims = ["accuracy", "completeness", "specificity", "synthesis"]
dim_kr = {"accuracy": "정확성", "completeness": "완전성",
          "specificity": "구체성", "synthesis": "통합성"}

agg = {m: {d: [] for d in dims} for m in method_names}
for r in all_results:
    for m in method_names:
        s = r["scores"][m]
        for d in dims:
            agg[m][d].append(getattr(s, d))

header = f"{'지표':<10}" + "".join(f" | {m:>14}" for m in method_names)
print(header)
print("─" * len(header))
for d in dims:
    vals = [f"{np.mean(agg[m][d]):.1f}" for m in method_names]
    print(f"{dim_kr[d]:<10}" + "".join(f" | {v:>14}" for v in vals))

print("─" * len(header))
overalls = []
for m in method_names:
    all_s = [v for d in dims for v in agg[m][d]]
    overalls.append(np.mean(all_s))
vals = [f"{o:.1f}" for o in overalls]
print(f"{'전체 평균':<10}" + "".join(f" | {v:>14}" for v in vals))

In [ ]:
# 질문 유형별 분석
print("\n질문 유형별 평균 점수:")
type_scores = {}
for r in all_results:
    qtype = r["question"]["type"]
    if qtype not in type_scores:
        type_scores[qtype] = {m: [] for m in method_names}
    for m in method_names:
        type_scores[qtype][m].append(r["scores"][m].average)

for qtype, scores in type_scores.items():
    print(f"\n  {qtype}:")
    for m in method_names:
        avg = np.mean(scores[m])
        bar = '█' * int(avg) + '░' * (10 - int(avg))
        print(f"    {m:<16}: {avg:.1f} {bar}")

In [ ]:
# 시간 비교
print("\n시간 비교:")
for m in method_names:
    times = [r[m]["time"] for r in all_results]
    print(f"  {m:<16}: 평균 {np.mean(times):.1f}s (구축 {build_times[m]:.1f}s)")

## 5. 토론 질문

1. **어떤 방법이 가장 좋을까?**  
   → 정답: "상황에 따라 다르다." 각각의 trade-off를 설명하세요.

2. **논문이 100편, 1000편이 되면?**  
   → RAG: 확장 쉬움 / Wiki: index.md가 커짐, 탐색 비용 증가

3. **실시간 업데이트가 필요하면?**  
   → RAG: 새 청크 추가만 하면 됨 / Wiki: 관련 페이지 모두 업데이트 필요

4. **인간 전문가와 협업한다면?**  
   → Wiki의 투명성이 큰 장점 (마크다운 직접 편집 가능)

5. **비용 최적화는?**  
   → Wiki: 높은 초기 비용, 낮은 쿼리 비용  
   → RAG: 낮은 초기 비용, 매 쿼리마다 임베딩 비용

## 6. 최종 과제

**자신만의 벤치마크 질문 3개를 만들고 비교하세요.**

- 단일 논문 팩트 질문 1개
- 크로스 논문 종합 질문 1개  
- 전체 논문 집계 질문 1개

결과를 리포트로 작성하여 제출하세요.